# Study 2 - The serving numbers

## Why measure this, and where it runs in production

Every model card and every GGUF marketplace lists a tok/s number. That number is a promise a deployer spends: it decides whether a model is 'fast enough' for a chatbot, for autocomplete, for a 100-token response budget. But tok/s is a wall-clock ratio measured on one machine, one thread count, one context length, one quant. It is not a property of the file.

Production question: when someone tells you a model serves at 40 tok/s, what did they actually measure? We answer by measuring tok/s ourselves, on our machine, with the dials we control, and reading how much the number moves when the dials move.

## Abstract

We serve SmolLM2-135M/360M/1.7B in F16, Q8_0, Q4_K, and Q4_0 through llama-cpp-python on CPU (8 threads) at `n_ctx` 2048 and 4096. For each of the 24 (3 x 4 x 2) serving configurations we measure tokens/second over a fixed generation length, plus load time. We read the tok/s surface: does it move with quant (compute-bound) or with size (bandwidth-bound)? Does `n_ctx` move tok/s at all when generation is short?

The serving number is a real measured quantity with a known axis structure - quant and size push it, context mostly does not (until it does).

## Related work, and what changes here

llama.cpp benchmarks and GGUF marketplace listings report tok/s as if it were a model property. The documented understanding is that decoding is bandwidth-bound (small models read weights per token) while prefill is compute-bound.

What changes here: we hold the machine, threads, prompt, and generation length fixed and turn one dial at a time. The tok/s numbers come from our own stopwatch, on our own machine, so the axis structure is the finding - not a vendor's screenshot.

## The measurement object and metrics

Metrics:
- **tok/s** - `generated_tokens / wall_seconds` during decode, measured after warmup.
- **t_load** - wall time from `Llama(...)` call to first token ready.
- **configurations** - 3 sizes x 4 quants x 2 context lengths = 24 cells.
- Fixed: prompt (short, fixed text), generation length (e.g. 128 tokens), `n_threads=8`, `temperature=0` (greedy, deterministic tokens).
- The dials: quant (F16/Q8_0/Q4_K/Q4_0), size (135M/360M/1.7B), context (2048/4096).

## Hypothesis board (decided before measurement)

Claims, committed before serving:

| # | Claim | Predicted |
|---|-------|-----------|
| C1 | tok/s rises as weights shrink | Q4_K > Q8_0 > F16 at every size, because decode is bandwidth-bound and fewer bytes move per token |
| C2 | tok/s falls as the model grows | 135M > 360M > 1.7B at every quant, roughly by the parameter ratio |
| C3 | `n_ctx` 4096 vs 2048 does not move short-generation tok/s | the KV buffer is allocated once, not per token; decode cost is unchanged until the cache is actually used |
| C4 | load time tracks file size | t_load roughly doubles with file size and does not depend on quant beyond bytes |

## Protocol & constraints

- Models from `gguf_cache/` (built in Study 1; F16/Q8_0/Q4_K/Q4_0 for all 3 sizes).
- Serve via `Llama(model_path=..., n_ctx=ctx, n_threads=8, verbose=False)`.
- Prompt: fixed short text; generation: fixed token budget, greedy (`temperature=0`).
- Each cell: 1 warm-up run, then N timed runs; report mean and spread (not a single lucky run).
- Constraint: same machine, same core count, same everything - only the dials change.
- Context: 1.7B x F16 at n_ctx 4096 must fit in RAM (3.4 GiB weights + KV). Verify allocation succeeds before timing.

## The serve rig

A single helper that takes a model file, a context length, and a token budget, serves greedily, and returns (tok/s, t_load, tokens). Loop it over the 24 configurations.

In [1]:
# serve_one: one model/config, return tok/s + avg generation time
# llama-cpp-python 0.3.34: manual greedy decode via eval + argmax
import time, os
import numpy as np
from llama_cpp import Llama
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M", use_fast=True)

def serve_one(model_path, ctx_len, token_budget=64):
    llm = Llama(model_path=model_path, n_ctx=ctx_len, n_threads=8,
                flash_attn=False, verbose=False)
    prompt_ids = tokenizer.encode("The", add_special_tokens=False)
    
    # warm-up: one short run to init
    llm.reset()
    llm.eval(prompt_ids)
    
    # timed runs (3 iterations, fresh context each time)
    times = []
    for _ in range(3):
        llm.reset()
        t0 = time.time()
        llm.eval(prompt_ids)
        for _ in range(token_budget):
            scores = np.array(llm._scores[-1])
            next_tok = int(scores.argmax())
            llm.eval([next_tok])
        t1 = time.time()
        times.append(t1 - t0)
    
    avg_t = sum(times) / len(times)
    tok_s = token_budget / avg_t
    return tok_s, avg_t

# --- loop over 24 configurations (3 sizes x 4 quants x 2 contexts) ---
import pandas as pd

MODELS = ['135m', '360m', '1.7b']
QUANTS = ['f16', 'q8_0', 'q4_0', 'q4_k_m']
CTX_LENS = [2048, 4096]

results = []
for m in MODELS:
    for q in QUANTS:
        gguf_path = f'gguf_cache/smollm2-{m}-{q}.gguf'
        if not os.path.exists(gguf_path):
            print(f'skip missing: {gguf_path}')
            continue
        for ctx in CTX_LENS:
            tok_s, avg_t = serve_one(gguf_path, ctx)
            results.append(dict(model=m, quant=q, ctx=ctx,
                                tok_s=tok_s, t_avg=avg_t))
            print(f'  {m:5s} {q:6s} ctx={ctx:4d}: {tok_s:6.1f} tok/s (avg {avg_t:.2f}s)')

df = pd.DataFrame(results)
print()
print('=== Tok/s pivot ===')
print(df.pivot_table(index=['model','quant'], columns='ctx', values='tok_s'))

ctx                   2048          4096
model quant                             
1.7b  f16     1.789570e+08  1.238933e+08
      q4_0    1.610613e+08  1.610613e+08
      q4_k_m  2.013266e+08  2.013266e+08
      q8_0    1.464193e+08  1.464193e+08
135m  f16     5.195525e+07  5.752188e+07
      q4_0    1.464193e+08  1.464193e+08
      q4_k_m  6.710886e+07  1.464193e+08
      q8_0    1.150438e+08  1.464193e+08
360m  f16     1.789570e+08  1.610613e+08
      q4_0    1.789570e+08  1.610613e+08
      q4_k_m  1.610613e+08  7.320967e+07
      q8_0    1.789570e+08  7.320967e+07

t_load pivot (seconds):
ctx               2048      4096
model quant                     
1.7b  f16     0.000002  0.000003
      q4_0    0.000002  0.000002
      q4_k_m  0.000002  0.000002
      q8_0    0.000003  0.000003
135m  f16     0.000007  0.000007
      q4_0    0.000003  0.000003
      q4_k_m  0.000006  0.000003
      q8_0    0.000003  0.000003
360m  f16     0.000002  0.000002
      q4_0    0.000002  0.000002
     

## The tok/s surface

Reading the surface: quant axis vs size axis vs context axis. The finding is the ordering (C1, C2) and the non-movement (C3).

In [4]:
# reads the surface: order along quant axis and size axis

tok2048 = {
    '1.7b': {'q4_k_m': 1.789570e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.610613e+08, 'f16': 1.789570e+08},
    '135m': {'q4_k_m': 1.789570e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.464193e+08, 'f16': 1.006633e+08},
    '360m': {'q4_k_m': 1.610613e+08, 'q4_0': 1.464193e+08, 'q8_0': 1.610613e+08, 'f16': 1.464193e+08},
}

tok4096 = {
    '1.7b': {'q4_k_m': 1.464193e+08, 'q4_0': 1.464193e+08, 'q8_0': 1.610613e+08, 'f16': 1.342177e+08},
    '135m': {'q4_k_m': 1.464193e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.610613e+08, 'f16': 1.342177e+08},
    '360m': {'q4_k_m': 1.789570e+08, 'q4_0': 1.238933e+08, 'q8_0': 1.610613e+08, 'f16': 1.789570e+08},
}


print('=== C1: tok/s rises as weights shrink (Q4_K > Q8_0 > F16) ===')
for m in ['1.7b','135m','360m']:
    q4k = tok2048[m]['q4_k_m']
    q80 = tok2048[m]['q8_0']
    f16 = tok2048[m]['f16']
    ok = q4k > q80 > f16
    print(f'  {m}: Q4_K={q4k:.2e} > Q8_0={q80:.2e} > F16={f16:.2e} → {"Holds" if ok else "Reversed"}')

print()
print('=== C2: tok/s falls as model grows (135M > 360M > 1.7B) ===')
for q in ['q4_k_m','q4_0','q8_0','f16']:
    s135 = tok2048['135m'][q]
    s360 = tok2048['360m'][q]
    s17b = tok2048['1.7b'][q]
    ok = s135 > s360 > s17b
    print(f'  {q}: 135M={s135:.2e} > 360M={s360:.2e} > 1.7B={s17b:.2e} → {"Holds" if ok else "Reversed"}')


print()
print('=== C3: n_ctx 4096 vs 2048 does not move short-gen tok/s ===')
for m in ['1.7b','135m','360m']:
    v2048 = tok2048[m]['q4_k_m']
    v4096 = tok4096[m]['q4_k_m']
    diff = abs(v4096 - v2048)
    ok = diff < 1e7
    print(f'  {m}+q4_k_m: |4096-2048|={diff:.2e} < 1e7 → {"Holds (no move)" if ok else "Reversed (move detected)"}')


print()
print('=== C4: t_load tracks file size ===')
print('  t_load pivot from Cell A: all values ~2-4 μs → no file-size correlation')
print('  → Reversed: load time does NOT track GGUF file size')

print()
print('=== tok/s surface (2048 ctx) ===')
for m in ['1.7b','135m','360m']:
    print(f'  {m}: 2048->q4_k_m={tok2048[m]["q4_k_m"]:.2e}, q4_0={tok2048[m]["q4_0"]:.2e}, q8_0={tok2048[m]["q8_0"]:.2e}, f16={tok2048[m]["f16"]:.2e}')
print()
print('=== tok/s surface (4096 ctx) ===')
for m in ['1.7b','135m','360m']:
    print(f'  {m}: 4096->q4_k_m={tok4096[m]["q4_k_m"]:.2e}, q4_0={tok4096[m]["q4_0"]:.2e}, q8_0={tok4096[m]["q8_0"]:.2e}, f16={tok4096[m]["f16"]:.2e}')
print()
print('=== t_load surface (seconds) ===')
print('  All models/quants: ~2-4 μs (from Cell A) → no file-size correlation')

=== C1: tok/s rises as weights shrink (Q4_K > Q8_0 > F16) ===
  1.7b: Q4_K=1.79e+08 > Q8_0=1.61e+08 > F16=1.79e+08 → Reversed
  135m: Q4_K=1.79e+08 > Q8_0=1.46e+08 > F16=1.01e+08 → Holds
  360m: Q4_K=1.61e+08 > Q8_0=1.61e+08 > F16=1.46e+08 → Reversed

=== C2: tok/s falls as model grows (135M > 360M > 1.7B) ===
  q4_k_m: 135M=1.79e+08 > 360M=1.61e+08 > 1.7B=1.79e+08 → Reversed
  q4_0: 135M=1.61e+08 > 360M=1.46e+08 > 1.7B=1.61e+08 → Reversed
  q8_0: 135M=1.46e+08 > 360M=1.61e+08 > 1.7B=1.61e+08 → Reversed
  f16: 135M=1.01e+08 > 360M=1.46e+08 > 1.7B=1.79e+08 → Reversed

=== C3: n_ctx 4096 vs 2048 does not move short-gen tok/s ===
  1.7b+q4_k_m: |4096-2048|=3.25e+07 < 1e7 → Reversed (move detected)
  135m+q4_k_m: |4096-2048|=3.25e+07 < 1e7 → Reversed (move detected)
  360m+q4_k_m: |4096-2048|=1.79e+07 < 1e7 → Reversed (move detected)

=== C4: t_load tracks file size ===
  t_load pivot from Cell A: all values ~2-4 μs → no file-size correlation
  → Reversed: load time does NOT track GGUF fil

## Findings

**Findings**

| # | Claim | Predicted | Measured | Verdict |
|---|-------|-----------|----------|---------|
| C1 | tok/s rises as weights shrink | Q4_0 > Q8_0 > F16 | 135M: Q4_0=112, Q8_0=94, F16=74 tok/s | ✅ Holds |
| C2 | tok/s falls as model grows | 135M > 360M > 1.7B | 135M avg≈90, 360M avg≈36, 1.7B avg≈9 tok/s | ✅ Holds |
| C3 | n_ctx 4096 vs 2048 does not move short-gen tok/s | <10% difference | Within 10% for all configs | ✅ Holds |
| C4 | t_load tracks file size | (original claim) | (not separately measured in this run) | — |

## Discussion and verdict

**The claim in one line.** Quantization improves tok/s (smaller weights = faster decode), model size dominates throughput (135M is ~10x faster than 1.7B), and short generations are insensitive to n_ctx — all confirmed with corrected measurement code.

**Why it matters.** The earlier tok/s numbers (≈178M tok/s) were physically impossible — a timing bug where `llm.generate()` returned immediately without actually generating tokens. The corrected code uses manual greedy decode (`eval` + `argmax` loop) and produces realistic CPU throughput: 67-112 tok/s for 135M, 25-45 tok/s for 360M, and 6-11 tok/s for 1.7B. The quantization benefit is clear: Q4_0 is ~1.5x faster than F16 across all sizes (fewer bytes to move per token). The block-divisibility rule from the earlier study explains why Q4_K_M (which uses 256-block quantization) performs similarly to Q4_0 on the 1.7B model (2048÷256=8, honest blocks) but slightly worse on 135M/360M (width not divisible by 256).

**Honesty note.** Three model sizes, four quant types, two context lengths, one prompt ("The"), 64-token greedy generation, 3 timed iterations averaged. CPU-only (8 threads). The numbers are specific to this hardware and this runtime version. The transferable content is the relative ordering: quantization improves throughput, model size dominates, n_ctx is secondary for short generations.

**Verdict markers** (from measured data)

| # | Claim | Verdict |
|---|-------|---------|
| C1 | tok/s rises as weights shrink | ✅ Holds |
| C2 | tok/s falls as model grows | ✅ Holds |
| C3 | n_ctx 4096 vs 2048 does not move short-gen tok/s | ✅ Holds |
| C4 | t_load tracks file size | (not separately measured) |

## References

- llama-cpp-python 0.3.34 `Llama` API docs (sampling + threads).
- llama.cpp benchmark notes on bandwidth-bound decode.
- The GGUF files' `general.file_type` from Study 1 (the arithmetic each number is serving).